In [0]:
%sql
CREATE DATABASE delta_live_lake;

https://www.mssqltips.com/sqlservertip/7456/delta-live-tables-databricks-framework-a-data-transformation-tool/

In [0]:
dbutils.fs.mount(
 source = "wasbs://container_name@storage_account_name.blob.core.windows.net",
 mount_point = "/mnt/data/",
 extra_configs = {"fs.azure.account.key.admiraldatastorage.blob.core.windows.net":"account_key"})


In [0]:
%sql
CREATE STREAMING LIVE TABLE yellow_trips
COMMENT "The raw NY yellow taxi trips table"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT * FROM cloud_files("/mnt/data/yellow-taxi-csv/", "csv",map("cloudFiles.inferColumnTypes", "true"));

In [0]:
%sql
CREATE STREAMING LIVE TABLE taxi_zones_lookup
COMMENT "The NY yellow taxi zone lookup"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT * FROM cloud_files("/mnt/data/ny_taxi_lookups/", "csv",map("cloudFiles.inferColumnTypes", "true"));

In [0]:
%sql
CREATE LIVE TABLE yellow_trips_metrics
COMMENT "Aggregation by pickup location"
TBLPROPERTIES ("quality" = "silver")
AS
SELECT date(t.tpep_pickup_datetime) as pickup_date,
       l1.Borough as pickup_location, 
       l2.Borough as dropoff_location, 
       avg(t.passenger_count) as passenger_count_avg,
       avg(t.total_amount) as total_amount_avg,
       max(t.total_amount) as total_amount_max
FROM LIVE.yellow_trips t
  join LIVE.taxi_zones_lookup l1
     on t.PULocationID = l1.LocationID
  join LIVE.taxi_zones_lookup l2
     on t.DOLocationID = l2.LocationID   
GROUP BY  date(t.tpep_pickup_datetime) ,
       l1.Borough , 
       l2.Borough